# FINS5557 project notebook

Our one notebook for exploring the data and developing the model. Code that works and gets reused (loading, features, training, SHAP, fairness, PSI) is moved into `src/`, and this notebook imports it, so the app and the notebook always use the same steps.

| Section | Status |
|---|---|
| 0. Setup | done |
| 1. Data: exploration and checks | done |
| 2. Features | to do |
| 3. Models: baseline vs random forest vs XGBoost | to do |
| 4. Calibration and threshold | to do |
| 5. Explainability (SHAP) and fairness | to do |

See `docs/BUILD_PLAN.md` for the full plan.

**Before you run:** activate `.venv`, pick it as this notebook's kernel (top right, **Select Kernel**), then run top to bottom. Clear outputs before committing.

## 0. Setup
### 0.1 Install packages
Installs `requirements-dev.txt` (which includes `requirements.txt`). Skip if you've already installed them.

In [ ]:
%pip install -q -r ../requirements-dev.txt

### 0.2 Imports and package versions
Prints each version, so a broken setup shows up here rather than half-way through.

In [ ]:
import sys
import warnings
from importlib.metadata import version
from pathlib import Path

# Make `src` importable whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.config import GOOGLE_API_KEY, LLM_BACKEND, PROTECTED, RANDOM_STATE, TARGET
from src.data import BILL_AMT, PAY_STATUS, clean, features_target, load_raw, split

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)

print("Python", sys.version.split()[0])
for pkg in ["pandas", "numpy", "scikit-learn", "xgboost", "shap", "streamlit", "python-dotenv", "joblib", "ucimlrepo"]:
    print(f"{pkg:15s} {version(pkg)}")

### 0.3 Settings check
Reports only whether an API key was found, never the key itself.

In [ ]:
print("Random seed:", RANDOM_STATE)
print("LLM backend:", LLM_BACKEND)
print("GOOGLE_API_KEY found:", bool(GOOGLE_API_KEY))

## 1. Data: exploration and checks

**Findings** (this becomes the Data section of the report):
- **Default rate:** 22.1% (6,636 of 30,000 customers default next month), identical (22.1%) in the train, validation and test splits. A model that always predicts "no default" would score 77.9% accuracy, so we judge models on AUC, KS and Brier score, and use class weights.
- **Strongest early signal:** last month's repayment status `PAY_0` (correlation 0.33 with default; the other `PAY_*` months follow, 0.26 down to 0.19). Customers 1+ months late default at 50.3%, vs 13.8% for everyone else; at 2+ months late it is 69.6%. Credit limit also matters: the lowest limit quintile (≤ NT$50k) defaults at 31.8%, the highest (> NT$270k) at 13.8%, consistent with lenders giving riskier customers lower limits.
- **Data issues and how we handle them:**
  - No missing values; schema checked by `validate_schema()`.
  - Undocumented codes: EDUCATION 0/5/6 (345 rows) → 4 "other", MARRIAGE 0 (54 rows) → 3 "other", done by `clean()`.
  - `PAY_*` values -2 and 0 aren't in the UCI documentation but cover 58% of `PAY_0`; kept as-is (read as "no spending" and "paid the minimum").
  - 590 customers have a negative last bill (credit balance) and 2,598 have a bill ≤ 0, so ratio features in Step 2 need safe division.
  - 35 exact duplicate rows (56 if the target is ignored, so some identical profiles have different outcomes). Kept, since there's no customer ID to prove they're the same person.
  - Amounts are highly skewed (`BILL_AMT1` from -165,580 to 964,511), so logistic regression needs scaling.
- **Fairness preview (raw default rates, before any model):**
  - Men default more than women (24.2% vs 20.8%).
  - Default rises as education falls (graduate school 19.2%, university 23.7%, high school 25.2%); the small "other" group is 7.1%.
  - Married 23.5% vs single 20.9%.
  - By age it is U-shaped: lowest at 30–39 (20.3%), highest at 60+ (28.3%, only 339 people).
  - Because these groups already differ in raw risk, a model that never sees SEX, MARRIAGE or AGE can still give them different approval rates through correlated inputs. Step 10 checks this with the disparate-impact ratio.

### 1.1 Load the data
The first run downloads it from UCI and saves `data/raw/uci_credit_default.csv` (gitignored). Later runs read the local file.

In [ ]:
raw = load_raw()
print(raw.shape)  # expect (30000, 24)
raw.head()

### 1.2 Target: how many customers default?
About 22% is expected. This imbalance is why the models use class weights, and why accuracy alone would be misleading (predicting "no default" for everyone scores 78%).

In [ ]:
raw[TARGET].value_counts(normalize=True).rename("share")

### 1.3 Data quality checks

In [ ]:
print("Missing values:", int(raw.isna().sum().sum()))
print("Duplicate rows:", int(raw.duplicated().sum()))
print("Customers with a negative bill (credit balance) last month:", int((raw["BILL_AMT1"] < 0).sum()))
print()
print("EDUCATION codes:", raw["EDUCATION"].value_counts().sort_index().to_dict())
print("MARRIAGE codes: ", raw["MARRIAGE"].value_counts().sort_index().to_dict())

EDUCATION 0, 5, 6 and MARRIAGE 0 are not in the UCI documentation, so `clean()` groups them into "other". Duplicates are kept: with no customer ID we can't tell whether they are the same person.

In [ ]:
df = clean(raw)
print("EDUCATION after cleaning:", df["EDUCATION"].value_counts().sort_index().to_dict())
print("MARRIAGE after cleaning: ", df["MARRIAGE"].value_counts().sort_index().to_dict())

### 1.4 Repayment status vs default
`PAY_0` is last month's status: -1 paid on time, 1 to 9 months late (-2 and 0 are usually read as no spending and paying the minimum). Expect this to be the strongest predictor.

In [ ]:
(df.groupby("PAY_0")[TARGET]
   .agg(customers="size", default_rate="mean")
   .round(3))

### 1.5 Amounts
Credit limits and bills are in NT dollars and very skewed, which matters for scaling in logistic regression (tree models don't mind).

In [ ]:
df[["LIMIT_BAL", "AGE", *BILL_AMT[:2], "PAY_AMT1"]].describe().T

In [ ]:
# Default rate by credit-limit quintile: do customers with lower limits default more?
df.groupby(pd.qcut(df["LIMIT_BAL"], 5), observed=True)[TARGET].mean().rename("default_rate")

### 1.6 Fairness preview
These attributes are **not** model inputs, but we audit the model's decisions by them later. Here are the raw default rates by group as a baseline.

In [ ]:
age_band = pd.cut(df["AGE"], [20, 30, 40, 50, 60, 80], right=False)
for name, group in [("SEX (1 = male, 2 = female)", df["SEX"]), ("EDUCATION", df["EDUCATION"]),
                    ("MARRIAGE", df["MARRIAGE"]), ("Age band", age_band)]:
    print(name)
    print(df.groupby(group, observed=True)[TARGET].agg(customers="size", default_rate="mean").round(3), "\n")

### 1.7 Train / validation / test split
Stratified 60/20/20, done before any model fitting. The default rate should be the same in all three parts.

In [ ]:
train, val, test = split(df)
pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "default_rate": [train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "validation", "test"])

In [ ]:
X_train, y_train = features_target(train)
print(X_train.shape[1], "model inputs; protected attributes removed:", PROTECTED)
list(X_train.columns)

## 2. Features
_To do: see `docs/BUILD_PLAN.md`._

## 3. Models: baseline vs random forest vs XGBoost
_To do: see `docs/BUILD_PLAN.md`._

## 4. Calibration and threshold
_To do: see `docs/BUILD_PLAN.md`._

## 5. Explainability (SHAP) and fairness
_To do: see `docs/BUILD_PLAN.md`._